In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U \
    "protobuf>=5.29,<6.0" \
    "bitsandbytes>=0.45.0" \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "datasets==3.1.0" \
    "sentencepiece"

In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import numpy as np
import re
import time
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


In [2]:
import os

ADAPTER_DIR = "/kaggle/input/datasets/anjaliii25/reasonforge-sft-adapter"

if os.path.exists(ADAPTER_DIR):
    files = os.listdir(ADAPTER_DIR)
    print(f"✓ Adapter found at {ADAPTER_DIR}")
    print(f"Files: {files}")
    # Check the key file exists
    if "adapter_model.safetensors" in files and "adapter_config.json" in files:
        print("✓ Required adapter files present")
    else:
        print("✗ Missing adapter_model.safetensors or adapter_config.json")
else:
    print(f"✗ {ADAPTER_DIR} not found — check Add Data")

✓ Adapter found at /kaggle/input/datasets/anjaliii25/reasonforge-sft-adapter
Files: ['adapter_model.safetensors', 'merges.txt', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json']
✓ Required adapter files present


In [3]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Base model loaded. Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded. Memory: 0.71 GB


In [4]:
def extract_answer(text):
    """Extract numerical answer from model output."""
    boxed = re.search(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        ans = boxed.group(1).strip()
        ans = ans.replace(',', '').replace('$', '').replace(' ', '')
        return ans
    nums = re.findall(r'-?\d+\.?\d*', text.replace(',', ''))
    return nums[-1] if nums else None

def extract_gsm8k_gold(text):
    return text.split('####')[-1].strip().replace(',', '')

def evaluate_model(model, tokenizer, dataset, n_samples, max_new_tokens=512, label=""):
    correct = 0
    results = []
    print(f"\nEvaluating {label} on {n_samples} problems...")
    start = time.time()
    
    for i, ex in enumerate(dataset.select(range(n_samples))):
        messages = [{"role": "user", "content": ex['question']}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = extract_answer(response)
        gold = extract_gsm8k_gold(ex['answer'])
        is_correct = pred == gold if pred else False
        correct += is_correct
        results.append({'idx': i, 'pred': pred, 'gold': gold, 'correct': is_correct, 'response': response})
        
        if (i+1) % 10 == 0:
            elapsed = time.time() - start
            eta = elapsed / (i+1) * (n_samples - i - 1)
            print(f"  [{i+1}/{n_samples}] correct: {correct}/{i+1} ({100*correct/(i+1):.1f}%) | ETA: {eta/60:.1f} min")
    
    accuracy = correct / n_samples
    print(f"\n{label}: {correct}/{n_samples} = {accuracy:.3f} (took {(time.time()-start)/60:.1f} min)")
    return accuracy, results

def bootstrap_ci(correct_flags, n_boot=1000, ci=0.95):
    correct_flags = np.array(correct_flags)
    samples = [np.mean(np.random.choice(correct_flags, len(correct_flags), replace=True)) for _ in range(n_boot)]
    return np.mean(correct_flags), np.percentile(samples, (1-ci)/2*100), np.percentile(samples, (1-(1-ci)/2)*100)

print("Eval harness ready")

Eval harness ready


In [5]:
gsm8k = load_dataset("gsm8k", "main", split="test")
print(f"GSM8K test: {len(gsm8k)} problems")
print(f"\nSample Q: {gsm8k[0]['question'][:150]}")
print(f"Sample A: {gsm8k[0]['answer'][-80:]}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test: 1319 problems

Sample Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the rem
Sample A: a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [6]:
N_SAMPLES = 200

base_acc, base_results = evaluate_model(
    base_model, tokenizer, gsm8k,
    n_samples=N_SAMPLES,
    max_new_tokens=512,
    label="BASE Qwen-2.5-1.5B"
)


Evaluating BASE Qwen-2.5-1.5B on 200 problems...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  [10/200] correct: 6/10 (60.0%) | ETA: 66.3 min
  [20/200] correct: 10/20 (50.0%) | ETA: 68.2 min
  [30/200] correct: 14/30 (46.7%) | ETA: 62.7 min
  [40/200] correct: 19/40 (47.5%) | ETA: 58.9 min
  [50/200] correct: 24/50 (48.0%) | ETA: 55.3 min
  [60/200] correct: 29/60 (48.3%) | ETA: 50.5 min
  [70/200] correct: 34/70 (48.6%) | ETA: 47.4 min
  [80/200] correct: 41/80 (51.2%) | ETA: 43.6 min
  [90/200] correct: 47/90 (52.2%) | ETA: 39.8 min
  [100/200] correct: 52/100 (52.0%) | ETA: 36.2 min
  [110/200] correct: 57/110 (51.8%) | ETA: 32.2 min
  [120/200] correct: 64/120 (53.3%) | ETA: 28.9 min
  [130/200] correct: 68/130 (52.3%) | ETA: 25.3 min
  [140/200] correct: 73/140 (52.1%) | ETA: 21.4 min
  [150/200] correct: 80/150 (53.3%) | ETA: 17.9 min
  [160/200] correct: 85/160 (53.1%) | ETA: 14.3 min
  [170/200] correct: 90/170 (52.9%) | ETA: 10.6 min
  [180/200] correct: 98/180 (54.4%) | ETA: 7.1 min
  [190/200] correct: 103/190 (54.2%) | ETA: 3.6 min
  [200/200] correct: 110/200 (55

In [8]:
sft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("SFT adapter loaded")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

sft_acc, sft_results = evaluate_model(
    sft_model, tokenizer, gsm8k,
    n_samples=N_SAMPLES,
    max_new_tokens=1500,
    label="SFT Qwen-2.5-1.5B (QLoRA)"
)

SFT adapter loaded
Memory: 0.78 GB

Evaluating SFT Qwen-2.5-1.5B (QLoRA) on 200 problems...
  [10/200] correct: 5/10 (50.0%) | ETA: 489.6 min
  [20/200] correct: 8/20 (40.0%) | ETA: 466.3 min
  [30/200] correct: 11/30 (36.7%) | ETA: 439.4 min
  [40/200] correct: 19/40 (47.5%) | ETA: 401.1 min
  [50/200] correct: 27/50 (54.0%) | ETA: 363.2 min
  [60/200] correct: 33/60 (55.0%) | ETA: 335.3 min
  [70/200] correct: 40/70 (57.1%) | ETA: 316.3 min
  [80/200] correct: 47/80 (58.8%) | ETA: 294.0 min
  [90/200] correct: 54/90 (60.0%) | ETA: 266.1 min
  [100/200] correct: 59/100 (59.0%) | ETA: 242.4 min
  [110/200] correct: 63/110 (57.3%) | ETA: 215.3 min
  [120/200] correct: 69/120 (57.5%) | ETA: 190.5 min
  [130/200] correct: 76/130 (58.5%) | ETA: 166.0 min
  [140/200] correct: 83/140 (59.3%) | ETA: 140.4 min
  [150/200] correct: 88/150 (58.7%) | ETA: 117.3 min
  [160/200] correct: 93/160 (58.1%) | ETA: 94.4 min
  [170/200] correct: 98/170 (57.6%) | ETA: 70.8 min
  [180/200] correct: 102/180 

In [9]:
base_flags = [r['correct'] for r in base_results]
sft_flags = [r['correct'] for r in sft_results]

base_mean, base_lo, base_hi = bootstrap_ci(base_flags)
sft_mean, sft_lo, sft_hi = bootstrap_ci(sft_flags)

print(f"\n{'='*60}")
print(f"GSM8K Results (n={N_SAMPLES})")
print('='*60)
print(f"Base: {base_mean:.3f}  95% CI [{base_lo:.3f}, {base_hi:.3f}]")
print(f"SFT:  {sft_mean:.3f}  95% CI [{sft_lo:.3f}, {sft_hi:.3f}]")
print(f"Absolute lift: +{(sft_mean-base_mean)*100:.1f} points")
if base_mean > 0:
    print(f"Relative lift: +{((sft_mean/base_mean)-1)*100:.1f}%")


GSM8K Results (n=200)
Base: 0.550  95% CI [0.485, 0.620]
SFT:  0.560  95% CI [0.485, 0.630]
Absolute lift: +1.0 points
Relative lift: +1.8%


In [10]:
import json, pickle, shutil

results_summary = {
    "n_samples": N_SAMPLES,
    "hardware": "Kaggle T4, 4-bit NF4",
    "base_accuracy": float(base_mean),
    "base_ci_95": [float(base_lo), float(base_hi)],
    "sft_accuracy": float(sft_mean),
    "sft_ci_95": [float(sft_lo), float(sft_hi)],
    "absolute_lift_points": float((sft_mean - base_mean) * 100),
    "training_loss_final": 0.558,
    "training_steps": 312,
}

with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

with open("/kaggle/working/eval_raw_results.pkl", "wb") as f:
    pickle.dump({"base": base_results, "sft": sft_results}, f)

print("Saved eval_results.json and eval_raw_results.pkl")
print("\nDownload BOTH from the Files panel (right sidebar) immediately.\n")
print(json.dumps(results_summary, indent=2))

Saved eval_results.json and eval_raw_results.pkl

Download BOTH from the Files panel (right sidebar) immediately.

{
  "n_samples": 200,
  "hardware": "Kaggle T4, 4-bit NF4",
  "base_accuracy": 0.55,
  "base_ci_95": [
    0.485,
    0.62
  ],
  "sft_accuracy": 0.56,
  "sft_ci_95": [
    0.4848750000000001,
    0.63
  ],
  "absolute_lift_points": 1.0000000000000009,
  "training_loss_final": 0.558,
  "training_steps": 312
}
